In [4]:
# --- BLOQUE 0: INFRAESTRUCTURA MÍNIMA ---
import sys
import subprocess
import gzip
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from scipy.optimize import curve_fit
from astropy.cosmology import Planck18 as cosmo

print("[SYSTEM] Iniciando Protocolo FTP DIRECTO (Sin APIs)...")

# --- BLOQUE 1: DESCARGA DE ARCHIVO CRUDO (.tdat.gz) ---
# Construimos la URL directa al volcado de la tabla fermilgrb
# Base: https://heasarc.gsfc.nasa.gov/FTP/heasarc/dbase/dump/
FTP_URL = "https://heasarc.gsfc.nasa.gov/FTP/heasarc/dbase/dump/heasarc_fermilgrb.tdat.gz"

print(f"[NET] Atacando URL estática: {FTP_URL}")

try:
    response = requests.get(FTP_URL)
    response.raise_for_status()
    print(f"[SUCCESS] Archivo descargado ({len(response.content)/1024:.1f} KB).")

    # Descomprimir en memoria
    with gzip.open(io.BytesIO(response.content), 'rt', encoding='utf-8') as f:
        content = f.read()

except Exception as e:
    print(f"[CRITICAL] Fallo en descarga FTP: {e}")
    # Fallback por si el nombre es ligeramente distinto (a veces no lleva el prefijo heasarc_)
    try:
        ALT_URL = "https://heasarc.gsfc.nasa.gov/FTP/heasarc/dbase/dump/fermilgrb.tdat.gz"
        print(f"[RETRY] Probando alternativa: {ALT_URL}")
        response = requests.get(ALT_URL)
        response.raise_for_status()
        with gzip.open(io.BytesIO(response.content), 'rt', encoding='utf-8') as f:
            content = f.read()
    except Exception as e2:
        print(f"[FAIL] Imposible obtener el archivo: {e2}")
        sys.exit()

# --- BLOQUE 2: PARSEO ARTESANAL DE TDAT ---
# Los archivos TDAT de HEASARC tienen mucha basura en el header.
# Buscamos la línea que empieza por "<DATA>" o donde terminan los comentarios.

print("[PROC] Parseando estructura TDAT...")

lines = content.split('\n')
header_row_index = -1
delimiter = None

# Find the first line that is likely the header (contains column names)
# Skip lines that are empty, comments, or metadata (like "table_name = ...")
for i, line in enumerate(lines):
    stripped_line = line.strip()
    if not stripped_line: continue
    if stripped_line.startswith('#'): continue
    if '=' in stripped_line: continue # Skip key-value metadata lines
    if stripped_line.lower().startswith('<header>'): continue # Specific to this file's first line
    if stripped_line.lower().startswith('line'): continue # Specific TDAT metadata

    # If we reach here, this line is a candidate for column names.
    # We check if it contains typical column delimiters or multiple words.
    if '|' in stripped_line and len(stripped_line.split('|')) > 1: # Check for pipe delimiter
        header_row_index = i
        delimiter = '|'
        break
    # If not pipe-delimited, check for multiple spaces (whitespace delimited)
    elif '  ' in stripped_line: # At least two spaces often means whitespace delimited
        header_row_index = i
        delimiter = r'\s+' # Use regex for one or more whitespace
        break
    # Fallback: if it's the first non-metadata line and no clear delimiter is found yet,
    # assume it's a whitespace-delimited header.
    elif header_row_index == -1 and len(stripped_line.split()) > 1: # If first content line and has multiple words, assume it's the header
        header_row_index = i
        delimiter = r'\s+' # Use regex for one or more whitespace
        break

[SYSTEM] Iniciando Protocolo FTP DIRECTO (Sin APIs)...
[NET] Atacando URL estática: https://heasarc.gsfc.nasa.gov/FTP/heasarc/dbase/dump/heasarc_fermilgrb.tdat.gz
[SUCCESS] Archivo descargado (67.1 KB).
[PROC] Parseando estructura TDAT...
